# Task 1 — LoRA fine-tune (Qwen2.5-0.5B) — Colab run (T4 GPU)

**Why this notebook exists:** the full training run does not fit on the
CPU-only VM it was developed on (see `README.md` — the model download
there was blocked/denied, so no local fine-tuned numbers exist yet).
This notebook runs the *exact same* `train.py` + `eval.py` on a free
Colab T4 GPU. It takes roughly **15–30 minutes** end to end.

## Steps (do them in order)
1. In Colab: **Runtime → Change runtime type → T4 GPU** (not CPU).
   Check it worked: run the GPU-check cell below — it must print a
   Tesla T4 and `torch.cuda.is_available() = True`. If it says CPU,
   change the runtime and try again.
2. **Get the code** — run the clone cell. It clones this assignment
   repo. If the repo is not public yet / the clone fails, use the
   fallback: upload the `task1-lora/` folder to Colab (Files panel →
   Upload) and skip to the install cell, adjusting the `%cd` path.
3. Run the **install** cell (torch, transformers, peft, accelerate).
   This takes ~2–4 minutes.
4. Run the **train** cell — LoRA, r=16, alpha=32, lr 2e-4, 3 epochs,
   800 examples. On a T4 this is roughly **10–20 minutes**.
5. Run the **eval** cell — evaluates base *and* fine-tuned models on
   the 100-example held-out test set (greedy decoding). ~2–5 minutes.
6. **Download `results/results_finetuned.json`** (and
   `results/results_base.json`) from the Files panel and send them
   back, so the README results table can be filled in with real
   numbers. Also download `results/failures.md`.
7. Do **not** commit or push anything from Colab; just return the two
   results JSON files.

No GPU / Colab account details, passwords or keys are needed.


In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU - STOP, switch runtime to T4 GPU")


In [ ]:
# --- Get the code: option A (clone the repo) ---
# Replace REPO_URL with the real repo URL (ask for it if unsure) —
# do NOT guess it. If the clone fails, use option B below.
REPO_URL = "https://github.com/Ganesh-123-maker/tndrr-aiml-assignment.git"
!git clone $REPO_URL
%cd tndrr-aiml-assignment/task1-lora
!ls -R | head -n 40
# Option B (no clone needed): upload the task1-lora/ folder via the
# Files panel, then run: %cd /content/task1-lora  (adjust the path)


In [ ]:
# --- Install dependencies (~2-4 min) ---
!pip install -q torch transformers peft accelerate datasets
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__)"


In [ ]:
# --- Sanity: data is already generated (800 train / 100 dev / 100 test) ---
!ls -lh data/ && wc -l data/*.jsonl && head -n 1 data/test.jsonl


In [ ]:
# --- Smoke test first (20 examples, ~1-2 min on T4) ---
# If this finishes cleanly, the full run below will work.
!python train.py --max-examples 20 --epochs 1 --output-dir /tmp/smoke-adapter --config-out /tmp/smoke-config.json


In [ ]:
# --- FULL TRAINING RUN (LoRA r=16, alpha=32, lr 2e-4, 3 epochs) ---
# ~10-20 min on a T4. Adapter is saved to adapter/ (not committed).
!python train.py
!ls -lh adapter/ && cat lora_config.json


In [ ]:
# --- EVALUATE base vs fine-tuned on the held-out test set (n=100) ---
!python eval.py
!cat results/results_base.json
!cat results/results_finetuned.json
!cat results/failures.md


In [ ]:
# --- What to send back ---
# Download these files from the Colab Files panel and return them:
#   results/results_base.json
#   results/results_finetuned.json
#   results/failures.md
#   lora_config.json  (contains the wall-clock time of the training run)
import json
print("BASE:     ", json.load(open("results/results_base.json")))
print("FINE-TUNED:", json.load(open("results/results_finetuned.json")))
